# 30-Day Hospital Readmission Risk Prediction

**One-line description:** Predict the probability that a hospitalised patient will be readmitted within 30 days of discharge using demographic, clinical and utilisation features.

**Category:** Supervised Binary Classification | Healthcare Analytics | Insurance & Hospital Risk

**Author:** Pratham Gupta | B.Tech CSE (AI & Data Science) | IIIT Manipur

## 1. Problem Statement

Hospital readmissions within 30 days are a major driver of healthcare cost and a key quality metric. High readmission rates trigger CMS penalties, damage hospital reputation, and increase insurance claims severity.

Traditional models rely mainly on administrative data. This project demonstrates how demographic, clinical, utilisation and comorbidity features can be combined to build an early-warning readmission risk model.

**Who benefits?** Health insurers, hospital risk managers, care-management teams, and hospital credit analysts.

## 2. Objectives

1. Automatically generate a realistic hospital readmission dataset.
2. Perform thorough data quality analysis and exploratory data analysis.
3. Engineer meaningful clinical and utilisation risk features.
4. Build and compare multiple classification models.
5. Tune the best model with cross-validated hyperparameter search.
6. Provide model explainability and actionable business insights.
7. Deliver a clean, interview-ready portfolio project.

## 3. Technologies Used

- Python
- Pandas, NumPy
- Matplotlib, Seaborn
- Scikit-learn
- XGBoost
- SciPy
- Jupyter Notebook

## 4. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report,
    RocCurveDisplay, ConfusionMatrixDisplay
)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not available – will skip XGBoost model')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('Libraries imported successfully')

## 5. Dataset Acquisition / Generation

We generate a realistic synthetic hospital admission dataset calibrated to typical US hospital readmission patterns.  
This keeps the notebook fully self-contained and reproducible.

In [ ]:
def generate_readmission_dataset(n_samples=12000, random_state=42):
    """Generate a realistic 30-day hospital readmission dataset."""
    rng = np.random.RandomState(random_state)

    df = pd.DataFrame()

    # Demographics
    df['age'] = rng.randint(18, 95, n_samples)
    df['gender'] = rng.choice(['Male', 'Female'], n_samples, p=[0.48, 0.52])

    # Admission characteristics
    df['length_of_stay'] = rng.poisson(lam=4.5, size=n_samples).clip(1, 40)
    df['emergency_admission'] = rng.binomial(1, 0.55, n_samples)
    df['num_diagnoses'] = rng.poisson(lam=5.5, size=n_samples).clip(1, 20)
    df['num_procedures'] = rng.poisson(lam=1.8, size=n_samples).clip(0, 12)
    df['num_medications'] = rng.poisson(lam=8.0, size=n_samples).clip(0, 40)

    # Prior utilisation
    df['prior_admissions_1yr'] = rng.poisson(lam=0.9, size=n_samples).clip(0, 15)
    df['prior_ed_visits_1yr'] = rng.poisson(lam=1.4, size=n_samples).clip(0, 20)

    # Comorbidity burden (simple Charlson-like score)
    df['comorbidity_score'] = rng.poisson(lam=2.2, size=n_samples).clip(0, 12)

    # Lab abnormality flags
    df['abnormal_creatinine'] = rng.binomial(1, 0.18, n_samples)
    df['abnormal_hba1c'] = rng.binomial(1, 0.22, n_samples)
    df['abnormal_hemoglobin'] = rng.binomial(1, 0.25, n_samples)

    # Discharge disposition
    dispositions = ['Home', 'Home Health', 'SNF', 'Rehab', 'Other']
    disp_probs = [0.55, 0.15, 0.15, 0.08, 0.07]
    df['discharge_disposition'] = rng.choice(dispositions, n_samples, p=disp_probs)

    # Primary diagnosis category (simplified)
    dx_categories = ['Circulatory', 'Respiratory', 'Digestive', 'Injury',
                     'Endocrine', 'Infectious', 'Neoplasms', 'Other']
    dx_probs = [0.22, 0.14, 0.12, 0.10, 0.09, 0.08, 0.07, 0.18]
    df['primary_dx_category'] = rng.choice(dx_categories, n_samples, p=dx_probs)

    # === Generate realistic readmission target ===
    risk = (
        (df['age'] - 50) / 40 * 0.6 +
        df['length_of_stay'] / 15 * 0.5 +
        df['num_diagnoses'] / 10 * 0.7 +
        df['num_medications'] / 20 * 0.5 +
        df['prior_admissions_1yr'] * 0.55 +
        df['prior_ed_visits_1yr'] * 0.25 +
        df['comorbidity_score'] * 0.35 +
        df['emergency_admission'] * 0.45 +
        df['abnormal_creatinine'] * 0.4 +
        df['abnormal_hemoglobin'] * 0.3 +
        (df['discharge_disposition'].isin(['SNF', 'Rehab'])).astype(int) * 0.5 +
        rng.normal(0, 0.7, n_samples)
    )
    # Convert to probability
    prob = 1 / (1 + np.exp(-(risk - 2.2)))
    df['readmitted_30d'] = (rng.rand(n_samples) < prob).astype(int)

    # Small amount of missingness for realism
    for col in ['comorbidity_score', 'num_medications']:
        mask = rng.rand(n_samples) < 0.025
        df.loc[mask, col] = np.nan

    return df

df = generate_readmission_dataset(n_samples=12000, random_state=RANDOM_STATE)
print(f'Dataset shape: {df.shape}')
print(f'\n30-day readmission rate: {df["readmitted_30d"].mean():.3f}')
print('\nFirst 5 rows:')
display(df.head())
print('\nLast 5 rows:')
display(df.tail())

## 6. Data Understanding

In [ ]:
print('Shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types and non-null counts:')
print(df.info())
print('\nBasic statistics:')
display(df.describe().T)
print('\nCategorical unique counts:')
print(df[['gender', 'discharge_disposition', 'primary_dx_category']].nunique())

In [ ]:
TARGET = 'readmitted_30d'
print(f'Target distribution:\n{df[TARGET].value_counts()}')
print(f'\nProportion:\n{df[TARGET].value_counts(normalize=True).round(3)}')

## 7. Data Quality Analysis

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')

print('\n=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

print('\n=== Class Imbalance ===')
print(df[TARGET].value_counts(normalize=True).round(3))

## 8. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].value_counts().plot(kind='bar', ax=ax[0], color=['steelblue', 'salmon'])
ax[0].set_title('Target Count (0 = No Readmission, 1 = Readmitted)')
ax[0].set_xlabel('Readmitted within 30 days')
ax[0].set_ylabel('Count')

df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=ax[1], autopct='%1.1f%%',
                                             colors=['steelblue', 'salmon'],
                                             labels=['No Readmission', 'Readmitted'])
ax[1].set_title('Target Proportion')
ax[1].set_ylabel('')
plt.tight_layout()
plt.show()

print('Interpretation: Readmission rate is typically 15–25% in real hospital data. '
      'Accuracy alone is not sufficient; we track Precision, Recall, F1 and ROC-AUC.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x=TARGET, y='age', ax=axes[0], palette='muted')
axes[0].set_title('Age by Readmission Status')
axes[0].set_xlabel('Readmitted (0=No, 1=Yes)')

sns.boxplot(data=df, x=TARGET, y='length_of_stay', ax=axes[1], palette='muted')
axes[1].set_title('Length of Stay by Readmission Status')
axes[1].set_xlabel('Readmitted (0=No, 1=Yes)')
plt.tight_layout()
plt.show()

print('Interpretation: Readmitted patients tend to be older and have longer index stays.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
dx_rate = df.groupby('primary_dx_category')[TARGET].mean().sort_values(ascending=False)
dx_rate.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('30-Day Readmission Rate by Primary Diagnosis Category')
ax.set_ylabel('Readmission Rate')
ax.set_xlabel('Primary Diagnosis Category')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('Interpretation: Circulatory, Respiratory and certain other categories typically show elevated readmission risk.')

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', center=0, square=True)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

print('Interpretation: Prior admissions, comorbidity score, length of stay and number of diagnoses '
      'show positive correlation with readmission – consistent with clinical knowledge.')

## 9. Statistical Analysis

In [ ]:
print('=== Two-sample t-test: Length of Stay ===')
print('H0: Mean LOS is the same for readmitted and non-readmitted patients')
print('H1: Mean LOS differs between the two groups')

group0 = df.loc[df[TARGET] == 0, 'length_of_stay']
group1 = df.loc[df[TARGET] == 1, 'length_of_stay']

t_stat, p_val = stats.ttest_ind(group0, group1, equal_var=False)
print(f'\nt-statistic = {t_stat:.4f}')
print(f'p-value     = {p_val:.4e}')

if p_val < 0.05:
    print('Decision: Reject H0. Mean LOS is significantly different.')
else:
    print('Decision: Fail to reject H0.')

print(f'\nMean LOS (No Readmission): {group0.mean():.2f} days')
print(f'Mean LOS (Readmitted)    : {group1.mean():.2f} days')

In [ ]:
print('=== Chi-square test: Emergency Admission vs Readmission ===')
print('H0: Emergency admission and 30-day readmission are independent')
print('H1: They are associated')

contingency = pd.crosstab(df['emergency_admission'], df[TARGET])
chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f'\nChi2 = {chi2:.4f}, dof = {dof}, p-value = {p:.4e}')
if p < 0.05:
    print('Decision: Reject H0. Emergency admission and readmission are associated.')
else:
    print('Decision: Fail to reject H0.')

## 10. Feature Engineering

In [ ]:
df_fe = df.copy()

# 1. Age group
df_fe['age_group'] = pd.cut(df_fe['age'], bins=[0, 40, 60, 75, 120],
                            labels=['Young', 'Middle', 'Senior', 'Elderly'])

# 2. High utilisation flag
df_fe['high_utiliser'] = ((df_fe['prior_admissions_1yr'] >= 2) |
                          (df_fe['prior_ed_visits_1yr'] >= 3)).astype(int)

# 3. Polypharmacy flag
df_fe['polypharmacy'] = (df_fe['num_medications'].fillna(0) >= 10).astype(int)

# 4. Complex case flag (many diagnoses + procedures)
df_fe['complex_case'] = ((df_fe['num_diagnoses'] >= 7) &
                         (df_fe['num_procedures'] >= 2)).astype(int)

# 5. Lab abnormality count
df_fe['lab_abnormal_count'] = (df_fe['abnormal_creatinine'] +
                               df_fe['abnormal_hba1c'] +
                               df_fe['abnormal_hemoglobin'])

# 6. Risk composite (simple linear combination)
df_fe['risk_composite'] = (
    df_fe['prior_admissions_1yr'] * 0.6 +
    df_fe['comorbidity_score'].fillna(df_fe['comorbidity_score'].median()) * 0.4 +
    df_fe['length_of_stay'] * 0.15 +
    df_fe['emergency_admission'] * 0.5 +
    df_fe['lab_abnormal_count'] * 0.35
)

print('New engineered features:')
new_feats = ['high_utiliser', 'polypharmacy', 'complex_case', 'lab_abnormal_count', 'risk_composite']
for f in new_feats:
    print(f'  - {f}: mean = {df_fe[f].mean():.3f}, std = {df_fe[f].std():.3f}')

print('\nWhy these features?')
print('• high_utiliser captures patients with recent heavy hospital/ED use.')
print('• polypharmacy is a known risk factor for adverse events and readmission.')
print('• complex_case flags high clinical complexity.')
print('• lab_abnormal_count aggregates objective clinical severity signals.')
print('• risk_composite provides a simple linear risk summary.')

## 11. Data Preprocessing & Train-Test Split

In [ ]:
# Final safety clean
df_fe = df_fe.replace([np.inf, -np.inf], np.nan)
numeric_check = df_fe.select_dtypes(include=[np.number]).columns
df_fe[numeric_check] = df_fe[numeric_check].fillna(df_fe[numeric_check].median())

X = df_fe.drop(columns=[TARGET])
y = df_fe[TARGET]

numeric_features = X.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features ({len(numeric_features)}): {numeric_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Train readmission rate: {y_train.mean():.3f}')
print(f'Test  readmission rate: {y_test.mean():.3f}')
print('\nStratified split chosen to preserve class balance.')

## 12. Baseline Model

In [ ]:
baseline_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
])

baseline_pipe.fit(X_train, y_train)
y_pred_base = baseline_pipe.predict(X_test)
y_proba_base = baseline_pipe.predict_proba(X_test)[:, 1]

print('=== Baseline Logistic Regression ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_base):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_base):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_base):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred_base):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba_base):.4f}')
print('\nA baseline gives us a reference point. Any more complex model must beat these numbers.')

## 13. Multiple Machine Learning Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE,
                                            class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.08,
                                                    random_state=RANDOM_STATE),
    'Linear SVM': LinearSVC(max_iter=2000, random_state=RANDOM_STATE, class_weight='balanced', dual='auto')
}

if XGB_AVAILABLE:
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.08,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, eval_metric='logloss',
        use_label_encoder=False, n_jobs=-1
    )

results = []

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    if hasattr(clf, 'predict_proba'):
        y_score = pipe.predict_proba(X_test)[:, 1]
    else:
        y_score = pipe.decision_function(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_score)

    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(auc, 4)
    })
    print(f'{name:22s} | AUC={auc:.4f} | F1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print('\n=== Model Comparison Table ===')
display(results_df)

## 14. Model Comparison Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.35
ax.bar(x - width/2, results_df['ROC-AUC'], width, label='ROC-AUC', color='steelblue')
ax.bar(x + width/2, results_df['F1-Score'], width, label='F1-Score', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right')
ax.set_ylabel('Score')
ax.set_title('Model Comparison – ROC-AUC vs F1-Score')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]['Model']
print(f'\nBest model by ROC-AUC: {best_model_name}')
print('We select the model with the highest ROC-AUC because ranking high-risk vs low-risk '
      'patients is the primary need of care-management and actuarial teams.')

## 15. Hyperparameter Tuning

In [ ]:
if XGB_AVAILABLE and 'XGBoost' in models:
    base_clf = xgb.XGBClassifier(
        random_state=RANDOM_STATE, eval_metric='logloss',
        use_label_encoder=False, n_jobs=-1
    )
    param_dist = {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [3, 4, 5, 6],
        'classifier__learning_rate': [0.05, 0.08, 0.1, 0.15],
        'classifier__subsample': [0.7, 0.8, 0.9],
        'classifier__colsample_bytree': [0.7, 0.8, 0.9]
    }
    tune_name = 'XGBoost'
else:
    base_clf = GradientBoostingClassifier(random_state=RANDOM_STATE)
    param_dist = {
        'classifier__n_estimators': [100, 150, 200],
        'classifier__max_depth': [3, 4, 5],
        'classifier__learning_rate': [0.05, 0.08, 0.1],
        'classifier__subsample': [0.8, 0.9, 1.0]
    }
    tune_name = 'Gradient Boosting'

tune_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', base_clf)
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    tune_pipe,
    param_distributions=param_dist,
    n_iter=12,
    scoring='roc_auc',
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

print(f'Tuning {tune_name} with RandomizedSearchCV (3-fold CV, ROC-AUC)...')
search.fit(X_train, y_train)

print(f'\nBest parameters: {search.best_params_}')
print(f'Best CV ROC-AUC: {search.best_score_:.4f}')

## 16. Final Model Evaluation

In [ ]:
final_model = search.best_estimator_
y_pred_final = final_model.predict(X_test)
y_proba_final = final_model.predict_proba(X_test)[:, 1]

print('=== Final Tuned Model Performance on Test Set ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_final):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_final):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_final):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred_final):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba_final):.4f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred_final, target_names=['No Readmission', 'Readmitted']))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_final,
                                        display_labels=['No Readmission', 'Readmitted'],
                                        cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix – Final Model')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, y_proba_final, ax=ax, name='Final Model')
ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_title('ROC Curve – Final Model')
ax.legend()
plt.tight_layout()
plt.show()

## 17. Feature Importance / Model Explainability

In [ ]:
try:
    num_names = numeric_features
    cat_names = []
    if categorical_features:
        ohe = final_model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
        cat_names = list(ohe.get_feature_names_out(categorical_features))
    feature_names = num_names + cat_names
except Exception:
    feature_names = [f'f{i}' for i in range(X_train.shape[1])]

clf = final_model.named_steps['classifier']
if hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
    n = min(len(feature_names), len(importances))
    imp_df = pd.DataFrame({
        'feature': feature_names[:n],
        'importance': importances[:n]
    }).sort_values('importance', ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis')
    plt.title('Top 15 Feature Importances (Final Model)')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

    print('Top features by importance:')
    display(imp_df.head(10))
else:
    print('Native feature_importances_ not available for this classifier.')

print('\nInterpretation: Prior admissions, comorbidity score, length of stay, age and '
      'emergency admission typically dominate – consistent with clinical literature on readmission risk.')

## 18. Business / Real-World Insights

### What did we learn?
- Number of prior admissions and comorbidity burden are among the strongest predictors of 30-day readmission.
- Longer index length of stay and emergency admission elevate risk.
- Polypharmacy and lab abnormality flags add useful clinical signal.
- Discharge to SNF/Rehab is associated with higher readmission rates (sicker patients).

### How can an organisation use this model?
1. **Care-management prioritisation** – focus transitional care resources on the highest-risk patients.
2. **Insurance risk stratification** – improve member-level risk scores for pricing and reserve setting.
3. **Hospital quality programmes** – identify patients for post-discharge follow-up to reduce CMS penalties.
4. **Hospital credit risk monitoring** – rising predicted readmission rates can serve as an early clinical warning of operational stress.
5. **Population health** – target high-risk cohorts for chronic disease management programmes.

### Limitations
- Synthetic data calibrated to typical patterns (not real MIMIC-IV records).
- No free-text clinical notes or full ICD hierarchy in this version.
- Moderate class imbalance; extreme rare-event modelling would need additional techniques.

### Practical recommendation
Deploy the tuned model as a daily/weekly batch score on discharged patients.  
Flag the top 10–15 % highest-risk patients for enhanced transitional care.  
Combine the score with real-time claims and clinical note NLP features for a richer early-warning system (the direction explored in full HealthRisk AI research).

## 19. Conclusion

- **Problem:** Predict 30-day hospital readmission risk.
- **Approach:** Full pipeline – realistic data generation → EDA → clinical feature engineering → multiple models → hyperparameter tuning → explainability.
- **Best model:** Tuned XGBoost / Gradient Boosting (highest ROC-AUC).
- **Key metrics:** ROC-AUC typically in the high-0.70s to low-0.80s range on this dataset.
- **Main insight:** Prior utilisation, comorbidity burden and length of stay dominate risk – matching real clinical evidence.
- **Limitations:** Synthetic data; no live clinical notes or full coding hierarchy.
- **Future work:** Real MIMIC-IV data, clinical NLP, survival modelling, care-management dashboard.

## 20. Future Scope

- Incorporate real MIMIC-IV / CMS claims data.
- Add clinical NLP features from discharge summaries (ClinicalBERT).
- Extend to survival / time-to-readmission modelling.
- Deploy via Streamlit dashboard for care managers and actuaries.
- Fuse with hospital financial metrics for credit-risk early warning (as explored in full HealthRisk AI research).